# Environment setup

The next cell is optional if you have already installed the repository dependencies.

In [ ]:
%pip install -r requirements.txt

# Import libraries and set configuration

In [ ]:
# Import libraries and configure paths.
from pathlib import Path
import warnings
import time
import gc
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier
import shap
import joblib

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

dataset_folder = Path("data")

macro_file = (
    dataset_folder /
    "V4FinBench_Macroeconomic_Variables_2006_2021.csv"
)

results_folder = Path("results")

COMPANY_ID = "emis_id"
COMPANY_NAME = "company"
COUNTRY_COLUMN = "country"
YEAR_COLUMN = "year"
TARGET_COLUMN = "main_label"
LINK_COLUMN = "link"

print("Dataset folder exists:", dataset_folder.exists())
print("Macro file exists:", macro_file.exists())
print("Results folder:", results_folder)

if not dataset_folder.exists():
    raise FileNotFoundError(f"Place the V4FinBench input files in {dataset_folder.resolve()}.")

if not macro_file.exists():
    raise FileNotFoundError(f"Missing macroeconomic input: {macro_file.resolve()}")

results_folder.mkdir(
    parents=True,
    exist_ok=True
)

# leakage controls 

In [ ]:
# LEAKAGE AND FEATURE-CONTAMINATION CONTROLS

TARGET_COLUMN = "main_label"
COMPANY_ID = "emis_id"
YEAR_COLUMN = "year"

# Identifiers and descriptive fields that should not be predictors
IDENTIFIER_COLUMNS = [
    "num",
    "country",
    "company",
    "industry",
    "link",
    "emis_id",
    "_fold",
    "analysis_country_code"
]

# Variables that could contain information about the company's
# subsequent legal or operating condition
POTENTIAL_LEAKAGE_COLUMNS = [
    "operational_status",
    "Insolvency_flag"
]

# GDP-derived variables already present in V4FinBench.
# These must not enter the financial-only baseline.
EMBEDDED_MACRO_COLUMNS = [
    "Logarithm_of_total_assets/GDP",
    "Log_revenue/gdp",
    "Log_operating_profit/gdp",
    "Log_net_profit/gdp"
]

# The exchange-rate change contains a structural distortion
# associated with Slovakia's adoption of the euro.
EXCLUDED_MACRO_COLUMNS = [
    "macro_exchange_rate_change_pct_lag1"
]

# Date fields will not be used directly by the models.
# Company age may be calculated separately if required.
DATE_COLUMNS = [
    "incorporation_date_1",
    "incorporation_date_2"
]

def existing_columns(data, requested_columns):
    """Return only requested columns that exist in the dataset."""
    return [
        column
        for column in requested_columns
        if column in data.columns
    ]

print("Leakage-control definitions created.")

# Find and validate all H1–H6 files

In [ ]:
# Locate files for each horizon.
def find_horizon_files(horizon):

    complete_file = (
        dataset_folder /
        f"company_years_h{horizon}.parquet"
    )

    if complete_file.exists():
        return [complete_file]

    possible_folders = [
        dataset_folder / f"h{horizon}_parts",
        dataset_folder / f"H{horizon}_parts",
        dataset_folder
    ]

    files = []

    for folder in possible_folders:

        if folder.exists():

            files.extend(
                folder.glob(
                    f"company_years_h{horizon}_part_*.parquet"
                )
            )

    return sorted(set(files))


horizon_files = {}
validation_results = []

for horizon in range(1, 7):

    files = find_horizon_files(horizon)

    if not files:
        raise FileNotFoundError(
            f"No Parquet files found for H{horizon}."
        )

    reference_schema = None
    total_rows = 0

    for file in files:

        parquet = pq.ParquetFile(file)

        if reference_schema is None:
            reference_schema = parquet.schema_arrow

        elif parquet.schema_arrow != reference_schema:
            raise ValueError(
                f"Schema mismatch detected in {file.name}."
            )

        total_rows += parquet.metadata.num_rows

    horizon_files[horizon] = files

    validation_results.append(
        {
            "horizon": f"H{horizon}",
            "files": len(files),
            "rows": total_rows,
            "columns": len(reference_schema),
            "schemas_match": "Yes"
        }
    )

validation_table = pd.DataFrame(validation_results)

display(validation_table)

validation_table.to_csv(
    results_folder / "horizon_file_validation.csv",
    index=False
)

In [ ]:
# Release memory between processing stages.
import gc
import matplotlib.pyplot as plt

large_objects = [
    "h3_data",
    "h3_prepared",
    "h3_model_data",
    "shap_data",
    "complete_training_data",
    "complete_validation_data",
    "complete_testing_data",
    "development_training_data",
    "development_validation_data",
    "development_testing_data",
    "training_data",
    "validation_data",
    "testing_data",
    "X_train",
    "X_validation",
    "X_test",
    "X_train_financial",
    "X_validation_financial",
    "X_test_financial",
    "X_train_hybrid",
    "X_validation_hybrid",
    "X_test_hybrid",
    "all_predictions",
    "best_predictions",
    "prediction_parts",
    "financial_models",
    "hybrid_models",
    "shap_values",
    "X_transformed",
    "X_explain"
]

for object_name in large_objects:

    if object_name in globals():

        del globals()[object_name]

plt.close("all")

collected_objects = gc.collect()

print(
    "Memory cleanup completed."
)

print(
    "Objects collected:",
    collected_objects
)

# Horizon-loading function

In [ ]:
# Load each horizon in batches.
def load_horizon(horizon, batch_size=50_000):
    """Load one horizon in bounded batches using Arrow-backed columns."""

    gc.collect()
    data_parts = []

    for file in horizon_files[horizon]:
        parquet_file = pq.ParquetFile(
            file,
            memory_map=True
        )

        for record_batch in parquet_file.iter_batches(
            batch_size=batch_size,
            use_threads=True
        ):
            # Arrow-backed pandas columns avoid an additional full-size
            # NumPy/object copy while the large horizon is loaded.
            data_parts.append(
                record_batch.to_pandas(
                    types_mapper=pd.ArrowDtype,
                    split_blocks=True
                )
            )

        del parquet_file

    if not data_parts:
        raise FileNotFoundError(
            f"No readable Parquet data were found for H{horizon}."
        )

    data = pd.concat(
        data_parts,
        ignore_index=True,
        copy=False
    )

    del data_parts
    gc.collect()

    memory_mb = data.memory_usage(
        index=True,
        deep=True
    ).sum() / (1024 ** 2)

    print(
        f"H{horizon} loaded: "
        f"{data.shape[0]:,} rows and "
        f"{data.shape[1]} columns"
    )
    print(f"Approximate H{horizon} memory: {memory_mb:,.1f} MB")

    return data


h3_data = load_horizon(3)

display(
    h3_data[
        [COMPANY_ID, COUNTRY_COLUMN, YEAR_COLUMN, TARGET_COLUMN]
    ].head()
)

# Data quality and preprocessing

In [ ]:
# Corrected formats and validated essential columns

In [ ]:
# Confirm all essential fields exist.
essential_columns = [
    COMPANY_ID,
    COMPANY_NAME,
    COUNTRY_COLUMN,
    YEAR_COLUMN,
    TARGET_COLUMN,
    LINK_COLUMN
]

missing_essential_columns = [
    column
    for column in essential_columns
    if column not in h3_data.columns
]

if missing_essential_columns:
    raise KeyError(
        f"Missing essential columns: {missing_essential_columns}"
    )

h3_data[YEAR_COLUMN] = pd.to_numeric(
    h3_data[YEAR_COLUMN],
    errors="coerce"
).astype("Int64")

h3_data[TARGET_COLUMN] = pd.to_numeric(
    h3_data[TARGET_COLUMN],
    errors="coerce"
).astype("Int64")

for column in [
    "incorporation_date_1",
    "incorporation_date_2"
]:

    if column in h3_data.columns:

        h3_data[column] = pd.to_datetime(
            h3_data[column],
            errors="coerce"
        )

print("Company identifier:", COMPANY_ID)
print("Year type:", h3_data[YEAR_COLUMN].dtype)
print("Target values:", sorted(h3_data[TARGET_COLUMN].dropna().unique()))
print("First year:", h3_data[YEAR_COLUMN].min())
print("Last year:", h3_data[YEAR_COLUMN].max())

if not set(
    h3_data[TARGET_COLUMN].dropna().unique()
).issubset({0, 1}):

    raise ValueError("The target is not binary.")

## Basic dataset overview


In [ ]:
# BASIC H3 DATASET INFORMATION

dataset_memory_mb = (
    h3_data.memory_usage(index=True, deep=True).sum()
    / (1024 ** 2)
)

basic_dataset_information = pd.DataFrame(
    {
        "Measure": [
            "Rows",
            "Columns",
            "Approximate memory (MB)",
            "Unique companies",
            "Countries",
            "First financial year",
            "Last financial year",
            "Financial years covered"
        ],
        "Value": [
            f"{h3_data.shape[0]:,}",
            f"{h3_data.shape[1]:,}",
            f"{dataset_memory_mb:,.1f}",
            f"{h3_data[COMPANY_ID].nunique(dropna=True):,}",
            f"{h3_data[COUNTRY_COLUMN].nunique(dropna=True):,}",
            int(h3_data[YEAR_COLUMN].min()),
            int(h3_data[YEAR_COLUMN].max()),
            int(h3_data[YEAR_COLUMN].nunique(dropna=True))
        ]
    }
)

display(basic_dataset_information)

dtype_summary = (
    h3_data.dtypes.astype(str)
    .value_counts()
    .rename_axis("Data type")
    .reset_index(name="Number of columns")
)

display(dtype_summary)


## Target and country distributions

Because financial distress is extremely rare, both the number and percentage of distressed observations are shown. Country counts are also reported to make the geographic composition of the dataset clear.


In [ ]:
# TARGET DISTRIBUTION

target_distribution = (
    h3_data[TARGET_COLUMN]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Target")
    .reset_index(name="Observations")
)

target_distribution["Label"] = target_distribution["Target"].map(
    {0: "Non-distress", 1: "Financial distress"}
).fillna("Missing")

target_distribution["Percentage"] = (
    target_distribution["Observations"]
    / len(h3_data)
    * 100
)

display(
    target_distribution[
        ["Target", "Label", "Observations", "Percentage"]
    ].round({"Percentage": 4})
)

# COUNTRY DISTRIBUTION
country_distribution = (
    h3_data[COUNTRY_COLUMN]
    .value_counts(dropna=False)
    .rename_axis("Country")
    .reset_index(name="Observations")
)

country_distribution["Percentage"] = (
    country_distribution["Observations"]
    / len(h3_data)
    * 100
)

display(country_distribution.round({"Percentage": 2}))


## Descriptive statistics

The numeric table summarises the distribution of financial variables.

In [ ]:
# NUMERIC DESCRIPTIVE STATISTICS

numeric_summary_columns = [
    column
    for column in h3_data.select_dtypes(include=np.number).columns
    if column not in ["num", COMPANY_ID, TARGET_COLUMN]
]

numeric_descriptive_statistics = (
    h3_data[numeric_summary_columns]
    .describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
    )
    .transpose()
    .reset_index()
    .rename(columns={"index": "Variable"})
)

numeric_descriptive_statistics.to_csv(
    results_folder / "h3_numeric_descriptive_statistics.csv",
    index=False
)

display(numeric_descriptive_statistics.head(25).round(3))

# CATEGORICAL DESCRIPTIVE STATISTICS
categorical_candidates = [
    column
    for column in h3_data.columns
    if column not in numeric_summary_columns
    and column not in [
        "num", COMPANY_ID, COMPANY_NAME, LINK_COLUMN,
        "industry", TARGET_COLUMN
    ]
    and not pd.api.types.is_datetime64_any_dtype(h3_data[column])
]

# Restrict the displayed summary to genuinely low-cardinality categories.
# This avoids expensive and unhelpful frequency tables for names or identifiers.
categorical_summary_columns = [
    column
    for column in categorical_candidates
    if h3_data[column].nunique(dropna=True) <= 100
]

categorical_rows = []

for column in categorical_summary_columns:
    non_missing_values = h3_data[column].dropna()
    most_common = non_missing_values.value_counts().head(1)

    categorical_rows.append(
        {
            "Variable": column,
            "Non-missing observations": int(non_missing_values.shape[0]),
            "Missing observations": int(h3_data[column].isna().sum()),
            "Unique values": int(non_missing_values.nunique()),
            "Most common value": (
                str(most_common.index[0]) if not most_common.empty else None
            ),
            "Most common frequency": (
                int(most_common.iloc[0]) if not most_common.empty else 0
            )
        }
    )

categorical_descriptive_statistics = pd.DataFrame(categorical_rows)

categorical_descriptive_statistics.to_csv(
    results_folder / "h3_categorical_descriptive_statistics.csv",
    index=False
)

display(categorical_descriptive_statistics)


# Check duplicates

In [ ]:
# Summarise duplicate record checks.
duplicate_summary = pd.DataFrame(
    {
        "check": [
            "Completely identical rows",
            "Company-name/year duplicates",
            "EMIS-ID/year duplicates",
            "Country/EMIS-ID/year duplicates"
        ],
        "duplicates": [
            h3_data.duplicated().sum(),

            h3_data.duplicated(
                subset=[
                    COMPANY_NAME,
                    YEAR_COLUMN
                ]
            ).sum(),

            h3_data.duplicated(
                subset=[
                    COMPANY_ID,
                    YEAR_COLUMN
                ]
            ).sum(),

            h3_data.duplicated(
                subset=[
                    COUNTRY_COLUMN,
                    COMPANY_ID,
                    YEAR_COLUMN
                ]
            ).sum()
        ]
    }
)

display(duplicate_summary)

# Inspect genuine EMIS-ID/year duplicates

In [ ]:
# Identify genuine duplicate company-years.
genuine_duplicates = (
    h3_data.loc[
        h3_data.duplicated(
            subset=[
                COMPANY_ID,
                YEAR_COLUMN
            ],
            keep=False
        )
    ]
    .sort_values([
        COMPANY_ID,
        YEAR_COLUMN
    ])
)

print(
    "Rows in repeated EMIS-ID/year groups:",
    len(genuine_duplicates)
)

if len(genuine_duplicates) > 0:

    display(
        genuine_duplicates[
            [
                "num",
                COMPANY_ID,
                COMPANY_NAME,
                COUNTRY_COLUMN,
                YEAR_COLUMN,
                TARGET_COLUMN
            ]
        ].head(30)
    )

# Check missing, infinite and constant values

In [ ]:
# MEMORY-EFFICIENT MISSING, INFINITE AND CONSTANT-VALUE CHECKS

missing_counts = pd.Series(
    {
        column: int(h3_data[column].isna().sum())
        for column in h3_data.columns
    },
    name="missing_count"
)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": missing_counts.div(len(h3_data)).mul(100)
})

missing_summary = (
    missing_summary.loc[missing_summary["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

print("Columns with missing values:", len(missing_summary))
display(missing_summary.head(50))
missing_summary.to_csv(results_folder / "h3_missing_values.csv")

essential_missing = pd.Series(
    {
        column: int(h3_data[column].isna().sum())
        for column in [
            COMPANY_ID, COUNTRY_COLUMN, YEAR_COLUMN, TARGET_COLUMN
        ]
    },
    name="missing_values"
)

print("\nMissing essential values:")
display(essential_missing.to_frame())

if essential_missing.sum() > 0:
    raise ValueError("Essential identifier or target values are missing.")

# Reading dtypes does not materialise or copy the underlying data.
numeric_columns = [
    column
    for column, dtype in h3_data.dtypes.items()
    if pd.api.types.is_numeric_dtype(dtype)
]

infinite_results = []

for column in numeric_columns:
    values = pd.to_numeric(h3_data[column], errors="coerce")
    numeric_array = values.to_numpy(
        dtype="float64",
        na_value=np.nan,
        copy=False
    )
    infinite_count = int(np.isinf(numeric_array).sum())

    if infinite_count > 0:
        infinite_results.append({
            "variable": column,
            "infinite_values": infinite_count
        })

infinite_summary = pd.DataFrame(infinite_results)
display(infinite_summary)

# Replace affected columns one at a time to avoid a large block copy.
for column in infinite_summary.get("variable", pd.Series(dtype=str)):
    h3_data[column] = h3_data[column].replace(
        [np.inf, -np.inf],
        np.nan
    )

constant_columns = [
    column
    for column in h3_data.columns
    if h3_data[column].nunique(dropna=False) <= 1
]

print("Numeric columns checked:", len(numeric_columns))
print("Constant columns:", constant_columns)

# Diagnose financial outliers

In [ ]:
excluded_numeric_columns = [
    "num",
    COMPANY_ID,
    COUNTRY_COLUMN,
    YEAR_COLUMN,
    TARGET_COLUMN
]

financial_numeric_columns = [
    column
    for column in numeric_columns
    if column not in excluded_numeric_columns
]

outlier_results = []

for column in financial_numeric_columns:

    values = pd.to_numeric(
        h3_data[column],
        errors="coerce"
    ).dropna()

    if values.empty:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outlier_count = (
        (values < lower_limit) |
        (values > upper_limit)
    ).sum()

    outlier_results.append(
        {
            "variable": column,
            "minimum": values.min(),
            "p01": values.quantile(0.01),
            "median": values.median(),
            "p99": values.quantile(0.99),
            "maximum": values.max(),
            "iqr_outliers": outlier_count,
            "outlier_percentage":
                outlier_count / len(values) * 100
        }
    )

outlier_summary = (
    pd.DataFrame(outlier_results)
    .sort_values(
        "outlier_percentage",
        ascending=False
    )
)

display(outlier_summary.head(50))

outlier_summary.to_csv(
    results_folder / "h3_outlier_summary.csv",
    index=False
)

# Country mapping and macroeconomic integration

# Verify country codes from EMIS links

In [ ]:
# Standardise country codes for merging.
h3_data["emis_country_code"] = (
    h3_data[LINK_COLUMN]
    .astype(str)
    .str.extract(
        r"[?&]pc=([A-Za-z]{2})",
        expand=False
    )
    .str.upper()
)

country_mapping_table = pd.crosstab(
    h3_data[COUNTRY_COLUMN],
    h3_data["emis_country_code"],
    margins=True
)

display(country_mapping_table)

country_mapping_consistency = (
    h3_data
    .dropna(
        subset=[
            COUNTRY_COLUMN,
            "emis_country_code"
        ]
    )
    .groupby(COUNTRY_COLUMN)[
        "emis_country_code"
    ]
    .nunique()
)

if (country_mapping_consistency > 1).any():
    raise ValueError(
        "A numerical country code maps to multiple countries."
    )

verified_country_mapping = (
    h3_data
    .dropna(
        subset=[
            COUNTRY_COLUMN,
            "emis_country_code"
        ]
    )
    .groupby(COUNTRY_COLUMN)[
        "emis_country_code"
    ]
    .agg(lambda values: values.mode().iloc[0])
    .to_dict()
)

print("Verified country mapping:")
print(verified_country_mapping)

# Load and validate macroeconomic data

In [ ]:
# Load and inspect macroeconomic data.
macro_data = pd.read_csv(macro_file)

macro_data["year"] = pd.to_numeric(
    macro_data["year"],
    errors="coerce"
).astype("Int64")

macro_data["iso3"] = (
    macro_data["iso3"]
    .astype(str)
    .str.strip()
    .str.upper()
)

iso3_to_iso2 = {
    "CZE": "CZ",
    "HUN": "HU",
    "POL": "PL",
    "SVK": "SK"
}

macro_data["analysis_country_code"] = (
    macro_data["iso3"]
    .map(iso3_to_iso2)
)

macro_duplicates = (
    macro_data
    .duplicated(
        subset=[
            "analysis_country_code",
            "year"
        ]
    )
    .sum()
)

print("Macroeconomic shape:", macro_data.shape)
print("Countries:", macro_data["analysis_country_code"].nunique())
print("First year:", macro_data["year"].min())
print("Last year:", macro_data["year"].max())
print("Duplicate country-years:", macro_duplicates)
print("Missing cells:", macro_data.isna().sum().sum())

assert macro_data["analysis_country_code"].nunique() == 4
assert macro_data["year"].min() == 2006
assert macro_data["year"].max() == 2021
assert macro_duplicates == 0
assert macro_data.isna().sum().sum() == 0

print("Macroeconomic validation: PASSED")

display(macro_data.head())

# Create one-year-lagged macroeconomic variables

In [ ]:
# Select the macroeconomic indicators.
macroeconomic_variables = [
    "gdp_growth_pct",
    "unemployment_pct",
    "inflation_cpi_pct",
    "private_credit_pct_gdp",
    "exchange_rate_change_pct",
    "gdp_per_capita_current_usd"
]

for variable in macroeconomic_variables:

    macro_data[variable] = pd.to_numeric(
        macro_data[variable],
        errors="coerce"
    )

macro_data = macro_data.sort_values(
    [
        "analysis_country_code",
        "year"
    ]
).copy()

lagged_macro_variables = []

for variable in macroeconomic_variables:

    lagged_column = (
        f"macro_{variable}_lag1"
    )

    macro_data[lagged_column] = (
        macro_data
        .groupby("analysis_country_code")[
            variable
        ]
        .shift(1)
    )

    lagged_macro_variables.append(
        lagged_column
    )

macro_data[
    "macro_log_gdp_per_capita_lag1"
] = np.log1p(
    macro_data[
        "macro_gdp_per_capita_current_usd_lag1"
    ]
)

lagged_macro_variables.remove(
    "macro_gdp_per_capita_current_usd_lag1"
)

lagged_macro_variables.append(
    "macro_log_gdp_per_capita_lag1"
)

print("Lagged macroeconomic variables:")

for variable in lagged_macro_variables:
    print(variable)

display(
    macro_data[
        [
            "analysis_country_code",
            "year"
        ] + lagged_macro_variables
    ].head(20)
)

# Prepare and merge horizon

In [ ]:
# Prepare each horizon for modelling.
def prepare_horizon_data(horizon):

    data = load_horizon(horizon)

    data[YEAR_COLUMN] = pd.to_numeric(
        data[YEAR_COLUMN],
        errors="coerce"
    ).astype("Int64")

    data[TARGET_COLUMN] = pd.to_numeric(
        data[TARGET_COLUMN],
        errors="coerce"
    ).astype("Int8")

    data["analysis_country_code"] = (
        data[LINK_COLUMN]
        .astype(str)
        .str.extract(
            r"[?&]pc=([A-Za-z]{2})",
            expand=False
        )
        .str.upper()
    )

    missing_country = (
        data["analysis_country_code"]
        .isna()
    )

    data.loc[
        missing_country,
        "analysis_country_code"
    ] = (
        data.loc[
            missing_country,
            COUNTRY_COLUMN
        ]
        .map(verified_country_mapping)
    )

    for date_column in [
        "incorporation_date_1",
        "incorporation_date_2"
    ]:

        if date_column in data.columns:

            parsed_date = pd.to_datetime(
                data[date_column],
                errors="coerce"
            )

            data[
                f"{date_column}_company_age"
            ] = (
                data[YEAR_COLUMN] -
                parsed_date.dt.year
            )

            data.loc[
                data[
                    f"{date_column}_company_age"
                ] < 0,
                f"{date_column}_company_age"
            ] = np.nan

    rows_before = len(data)

    data = data.merge(
        macro_data[
            [
                "analysis_country_code",
                "year"
            ] + lagged_macro_variables
        ],
        on=[
            "analysis_country_code",
            "year"
        ],
        how="left",
        validate="many_to_one",
        indicator=True
    )

    assert len(data) == rows_before

    merge_rate = (
        data["_merge"]
        .eq("both")
        .mean()
    )

    print(
        f"H{horizon} macro merge rate:",
        f"{merge_rate:.2%}"
    )

    if merge_rate < 0.95:

        display(
            data.loc[
                data["_merge"] != "both",
                [
                    "analysis_country_code",
                    YEAR_COLUMN
                ]
            ]
            .drop_duplicates()
        )

        raise ValueError(
            f"H{horizon} macro merge rate is below 95%."
        )

    data = data.drop(
        columns=["_merge"],
        errors="ignore"
    )

    data = data.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if data[
        [
            COMPANY_ID,
            YEAR_COLUMN,
            TARGET_COLUMN
        ]
    ].isna().any().any():

        raise ValueError(
            f"H{horizon} has missing essential values."
        )

    return data


h3_prepared = prepare_horizon_data(3)

print("Prepared H3 shape:", h3_prepared.shape)

display(
    h3_prepared[
        lagged_macro_variables
    ]
    .isna()
    .sum()
    .to_frame("missing_values")
)

# Exploratory data analysis

# Distress prevalence across H1–H6

In [ ]:
# Summarise all horizon datasets.
horizon_summary = []

for horizon in range(1, 7):

    data = load_horizon(horizon)

    target = pd.to_numeric(
        data[TARGET_COLUMN],
        errors="coerce"
    )

    horizon_summary.append(
        {
            "horizon": f"H{horizon}",
            "years_ahead": horizon - 1,
            "observations": len(data),
            "distressed": int((target == 1).sum()),
            "non_distressed": int((target == 0).sum()),
            "distress_rate": (target == 1).mean()
        }
    )

    del data
    gc.collect()

horizon_summary = pd.DataFrame(
    horizon_summary
)

display(horizon_summary)

horizon_summary.to_csv(
    results_folder / "target_summary_by_horizon.csv",
    index=False
)

plt.figure(figsize=(9, 5))

sns.barplot(
    data=horizon_summary,
    x="horizon",
    y="distress_rate",
    color="#3569A8"
)

plt.title("Financial-Distress Rate by Prediction Horizon")
plt.xlabel("Horizon")
plt.ylabel("Distress rate")

plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda value, position:
        f"{value * 100:.2f}%"
    )
)

plt.tight_layout()

plt.savefig(
    results_folder / "distress_rate_by_horizon.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Macroeconomic trends and correlations

In [ ]:
# Examine correlations between macro indicators.
macro_correlation = (
    macro_data[
        macroeconomic_variables
    ]
    .corr()
)

plt.figure(figsize=(10, 7))

sns.heatmap(
    macro_correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between Macroeconomic Indicators")
plt.tight_layout()

plt.savefig(
    results_folder / "macroeconomic_correlations.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

macro_long = macro_data.melt(
    id_vars=[
        "analysis_country_code",
        "year"
    ],
    value_vars=macroeconomic_variables,
    var_name="indicator",
    value_name="value"
)

chart = sns.relplot(
    data=macro_long,
    x="year",
    y="value",
    hue="analysis_country_code",
    col="indicator",
    col_wrap=2,
    kind="line",
    marker="o",
    facet_kws={"sharey": False},
    height=3.4,
    aspect=1.3
)

chart.fig.suptitle(
    "Macroeconomic Indicators Across V4 Countries",
    y=1.02
)

chart.savefig(
    results_folder / "macroeconomic_trends.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Feature engineering and model design

In [ ]:
# Define financial and hybrid features

In [ ]:
# Define variables excluded from modelling.
excluded_columns = [
    "num",
    COMPANY_ID,
    COMPANY_NAME,
    LINK_COLUMN,
    "industry",
    "incorporation_date_1",
    "incorporation_date_2",
    "emis_country_code",
    COUNTRY_COLUMN,
    TARGET_COLUMN,
    "_fold"
] + constant_columns


def create_feature_sets(data):

    financial_candidates = [
        column
        for column in data.columns
        if (
            column not in excluded_columns
            and column not in lagged_macro_variables
        )
    ]

    financial_features = []

    for column in financial_candidates:

        if pd.api.types.is_numeric_dtype(
            data[column]
        ):

            financial_features.append(column)

        elif (
            data[column]
            .nunique(dropna=True)
            <= 50
        ):

            financial_features.append(column)

    hybrid_features = (
        financial_features +
        [
            column
            for column in lagged_macro_variables
            if column in data.columns
        ]
    )

    return financial_features, hybrid_features


financial_features, hybrid_features = (
    create_feature_sets(h3_prepared)
)

print("Financial features:", len(financial_features))
print("Hybrid features:", len(hybrid_features))
print("Added macroeconomic features:", len(lagged_macro_variables))

# Create company-grouped folds

In [ ]:
# Create company-grouped validation folds.
def create_grouped_folds(
    data,
    number_of_folds=5
):

    folds = pd.Series(
        -1,
        index=data.index,
        dtype="int8"
    )

    for country_code, indices in (
        data.groupby(
            "analysis_country_code"
        ).groups.items()
    ):

        indices = np.asarray(
            list(indices)
        )

        country_data = data.loc[
            indices
        ]

        splitter = StratifiedGroupKFold(
            n_splits=number_of_folds,
            shuffle=True,
            random_state=RANDOM_STATE
        )

        for fold_number, (
            unused,
            held_indices
        ) in enumerate(
            splitter.split(
                country_data,
                country_data[TARGET_COLUMN],
                groups=country_data[COMPANY_ID]
            )
        ):

            folds.loc[
                indices[held_indices]
            ] = fold_number

    if (folds < 0).any():
        raise RuntimeError(
            "Some rows were not assigned to a fold."
        )

    company_fold_count = (
        pd.DataFrame(
            {
                "emis_id": data[COMPANY_ID],
                "fold": folds
            }
        )
        .groupby("emis_id")["fold"]
        .nunique()
        .max()
    )

    assert company_fold_count == 1

    return folds


h3_prepared["_fold"] = (
    create_grouped_folds(h3_prepared)
)

fold_summary = (
    h3_prepared
    .groupby("_fold")[TARGET_COLUMN]
    .agg(
        observations="size",
        distressed="sum",
        distress_rate="mean"
    )
    .reset_index()
)

display(fold_summary)

# Training-only winsorisation

In [ ]:
# Limit extreme numeric predictor values.
class QuantileWinsorizer(
    BaseEstimator,
    TransformerMixin
):

    def __init__(
        self,
        lower_quantile=0.01,
        upper_quantile=0.99
    ):

        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):

        X = np.asarray(X, dtype=float)

        self.lower_limits_ = np.nanquantile(
            X,
            self.lower_quantile,
            axis=0
        )

        self.upper_limits_ = np.nanquantile(
            X,
            self.upper_quantile,
            axis=0
        )

        return self

    def transform(self, X):

        X = np.asarray(X, dtype=float)

        return np.clip(
            X,
            self.lower_limits_,
            self.upper_limits_
        )

# Preprocessing and model functions

In [ ]:
# Build the preprocessing transformation pipeline.
def create_preprocessor(
    X,
    scale_numeric=False
):

    numeric_features = (
        X.select_dtypes(include=np.number)
        .columns
        .tolist()
    )

    categorical_features = [
        column
        for column in X.columns
        if column not in numeric_features
    ]

    numeric_steps = [
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "winsorizer",
            QuantileWinsorizer(
                lower_quantile=0.01,
                upper_quantile=0.99
            )
        )
    ]

    if scale_numeric:

        numeric_steps.append(
            (
                "scaler",
                StandardScaler()
            )
        )

    numeric_pipeline = Pipeline(
        numeric_steps
    )

    categorical_pipeline = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=20
                )
            )
        ]
    )

    return ColumnTransformer(
        [
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )


def create_models(X):

    return {
        "Logistic Regression":
            Pipeline(
                [
                    (
                        "preprocessor",
                        create_preprocessor(
                            X,
                            scale_numeric=True
                        )
                    ),
                    (
                        "model",
                        LogisticRegression(
                            max_iter=1000,
                            class_weight="balanced",
                            solver="saga",
                            n_jobs=-1,
                            random_state=RANDOM_STATE
                        )
                    )
                ]
            ),

        "Random Forest":
            Pipeline(
                [
                    (
                        "preprocessor",
                        create_preprocessor(
                            X,
                            scale_numeric=False
                        )
                    ),
                    (
                        "model",
                        RandomForestClassifier(
                            n_estimators=200,
                            min_samples_leaf=5,
                            max_features="sqrt",
                            class_weight="balanced_subsample",
                            n_jobs=-1,
                            random_state=RANDOM_STATE
                        )
                    )
                ]
            ),

        "XGBoost":
            Pipeline(
                [
                    (
                        "preprocessor",
                        create_preprocessor(
                            X,
                            scale_numeric=False
                        )
                    ),
                    (
                        "model",
                        XGBClassifier(
                            n_estimators=500,
                            learning_rate=0.05,
                            max_depth=6,
                            subsample=0.8,
                            colsample_bytree=0.8,
                            objective="binary:logistic",
                            eval_metric="logloss",
                            tree_method="hist",
                            n_jobs=-1,
                            random_state=RANDOM_STATE
                        )
                    )
                ]
            )
    }

# Threshold and metric functions

In [ ]:
# Select threshold using validation F1.
def select_threshold(
    actual,
    probability
):

    thresholds = np.linspace(
        0.01,
        0.99,
        199
    )

    scores = [
        f1_score(
            actual,
            probability >= threshold,
            zero_division=0
        )
        for threshold in thresholds
    ]

    best_position = int(
        np.argmax(scores)
    )

    return (
        float(thresholds[best_position]),
        float(scores[best_position])
    )


def calculate_metrics(
    actual,
    probability,
    threshold
):

    prediction = (
        probability >= threshold
    ).astype(int)

    matrix = confusion_matrix(
        actual,
        prediction,
        labels=[0, 1]
    )

    return {
        "threshold": threshold,
        "accuracy":
            accuracy_score(actual, prediction),
        "balanced_accuracy":
            balanced_accuracy_score(actual, prediction),
        "precision":
            precision_score(
                actual,
                prediction,
                zero_division=0
            ),
        "recall":
            recall_score(
                actual,
                prediction,
                zero_division=0
            ),
        "f1":
            f1_score(
                actual,
                prediction,
                zero_division=0
            ),
        "roc_auc":
            roc_auc_score(actual, probability),
        "pr_auc":
            average_precision_score(
                actual,
                probability
            ),
        "brier_score":
            brier_score_loss(
                actual,
                probability
            ),
        "tn": matrix[0, 0],
        "fp": matrix[0, 1],
        "fn": matrix[1, 0],
        "tp": matrix[1, 1]
    }

# Model training and evaluation

# Initial H3 development run

In [ ]:
# Set the negative sampling ratio.


NEGATIVE_TO_POSITIVE_RATIO = 5

MAXIMUM_VALIDATION_ROWS = 50_000
MAXIMUM_TEST_ROWS = 50_000

XGBOOST_ESTIMATORS = 75

print(
    "Negative-to-positive training ratio:",
    NEGATIVE_TO_POSITIVE_RATIO
)

print(
    "Maximum validation rows:",
    f"{MAXIMUM_VALIDATION_ROWS:,}"
)

print(
    "Maximum test rows:",
    f"{MAXIMUM_TEST_ROWS:,}"
)

print(
    "XGBoost estimators:",
    XGBOOST_ESTIMATORS
)

# Training-sample function

In [ ]:
# Sample training data after grouping.
def create_development_training_sample(
    training_data,
    negative_to_positive_ratio=5
):

    positive_data = (
        training_data.loc[
            training_data[
                TARGET_COLUMN
            ] == 1
        ]
        .copy()
    )

    negative_data = (
        training_data.loc[
            training_data[
                TARGET_COLUMN
            ] == 0
        ]
        .copy()
    )

    required_negative_rows = min(
        len(negative_data),
        len(positive_data) *
        negative_to_positive_ratio
    )

    sampled_negative_data = (
        negative_data.sample(
            n=required_negative_rows,
            random_state=RANDOM_STATE
        )
    )

    sampled_training_data = pd.concat(
        [
            positive_data,
            sampled_negative_data
        ],
        ignore_index=True
    )

    sampled_training_data = (
        sampled_training_data
        .sample(
            frac=1,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )

    print(
        "Positive training observations:",
        f"{len(positive_data):,}"
    )

    print(
        "Sampled negative observations:",
        f"{len(sampled_negative_data):,}"
    )

    print(
        "Total development training rows:",
        f"{len(sampled_training_data):,}"
    )

    return sampled_training_data

# Validation/test sampling function

In [ ]:
# Sample evaluation data without leakage.
def create_evaluation_sample(
    data,
    maximum_rows=50_000
):

    if len(data) <= maximum_rows:

        return data.copy()

    positive_data = (
        data.loc[
            data[
                TARGET_COLUMN
            ] == 1
        ]
        .copy()
    )

    negative_data = (
        data.loc[
            data[
                TARGET_COLUMN
            ] == 0
        ]
        .copy()
    )

    negative_rows_required = max(
        0,
        maximum_rows -
        len(positive_data)
    )

    negative_rows_required = min(
        negative_rows_required,
        len(negative_data)
    )

    sampled_negative_data = (
        negative_data.sample(
            n=negative_rows_required,
            random_state=RANDOM_STATE
        )
    )

    sampled_data = pd.concat(
        [
            positive_data,
            sampled_negative_data
        ],
        ignore_index=True
    )

    sampled_data = (
        sampled_data
        .sample(
            frac=1,
            random_state=RANDOM_STATE
        )
        .reset_index(drop=True)
    )

    return sampled_data

# Load and prepare H3

In [ ]:
# Set the development-test horizon.
DEVELOPMENT_HORIZON = 3
VALIDATION_FOLD = 0
TEST_FOLD = (VALIDATION_FOLD + 1) % 5

# h3_prepared was created earlier and already contains the macro merge.
h3_model_data = h3_prepared.copy()

# Convert Boolean indicators for compatibility with SimpleImputer.
for column in h3_model_data.select_dtypes(include=["bool", "boolean"]).columns:
    h3_model_data[column] = h3_model_data[column].astype("float32")

# Exclude boundary-year rows for which every lagged macro variable is missing.
development_macro_features = [
    column for column in h3_model_data.columns
    if column.startswith("macro_")
    and column not in EXCLUDED_MACRO_COLUMNS
]
development_missing_macro = (
    h3_model_data[development_macro_features].isna().all(axis=1)
)
print(
    "H3 rows excluded without lagged macro data:",
    f"{int(development_missing_macro.sum()):,}"
)
h3_model_data = h3_model_data.loc[~development_missing_macro].copy()

if "_fold" not in h3_model_data.columns:
    h3_model_data["_fold"] = create_grouped_folds(h3_model_data)

print("Prepared H3 dimensions:", h3_model_data.shape)
print("Validation fold:", VALIDATION_FOLD)
print("Test fold:", TEST_FOLD)


# Create train, validation and test partitions

In [ ]:
# Retain complete development observations.
complete_training_data = (
    h3_model_data.loc[
        ~h3_model_data[
            "_fold"
        ].isin(
            [
                VALIDATION_FOLD,
                TEST_FOLD
            ]
        )
    ]
    .copy()
)

complete_validation_data = (
    h3_model_data.loc[
        h3_model_data[
            "_fold"
        ] == VALIDATION_FOLD
    ]
    .copy()
)

complete_testing_data = (
    h3_model_data.loc[
        h3_model_data[
            "_fold"
        ] == TEST_FOLD
    ]
    .copy()
)

print(
    "Complete training rows:",
    f"{len(complete_training_data):,}"
)

print(
    "Complete validation rows:",
    f"{len(complete_validation_data):,}"
)

print(
    "Complete testing rows:",
    f"{len(complete_testing_data):,}"
)

# Verify company separation

In [ ]:
# Identify companies assigned to training.
training_companies = set(
    complete_training_data[
        COMPANY_ID
    ]
)

validation_companies = set(
    complete_validation_data[
        COMPANY_ID
    ]
)

testing_companies = set(
    complete_testing_data[
        COMPANY_ID
    ]
)

training_validation_overlap = (
    training_companies.intersection(
        validation_companies
    )
)

training_testing_overlap = (
    training_companies.intersection(
        testing_companies
    )
)

validation_testing_overlap = (
    validation_companies.intersection(
        testing_companies
    )
)

print(
    "Training-validation company overlap:",
    len(training_validation_overlap)
)

print(
    "Training-testing company overlap:",
    len(training_testing_overlap)
)

print(
    "Validation-testing company overlap:",
    len(validation_testing_overlap)
)

assert len(
    training_validation_overlap
) == 0

assert len(
    training_testing_overlap
) == 0

assert len(
    validation_testing_overlap
) == 0

print(
    "Company leakage check: PASSED"
)

# Create smaller development datasets

In [ ]:
# Build the sampled development dataset.
development_training_data = (
    create_development_training_sample(
        complete_training_data,
        negative_to_positive_ratio=
            NEGATIVE_TO_POSITIVE_RATIO
    )
)

development_validation_data = (
    create_evaluation_sample(
        complete_validation_data,
        maximum_rows=
            MAXIMUM_VALIDATION_ROWS
    )
)

development_testing_data = (
    create_evaluation_sample(
        complete_testing_data,
        maximum_rows=
            MAXIMUM_TEST_ROWS
    )
)

print(
    "\nDevelopment training rows:",
    f"{len(development_training_data):,}"
)

print(
    "Development validation rows:",
    f"{len(development_validation_data):,}"
)

print(
    "Development testing rows:",
    f"{len(development_testing_data):,}"
)

# Check target distributions

In [ ]:
# Display class balance by sample.
def display_target_distribution(
    data,
    dataset_name
):

    target_distribution = (
        data[
            TARGET_COLUMN
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            "main_label"
        )
        .reset_index(
            name="observations"
        )
    )

    target_distribution[
        "percentage"
    ] = (
        target_distribution[
            "observations"
        ]
        / len(data)
        * 100
    )

    print(
        f"\n{dataset_name}:"
    )

    display(
        target_distribution
    )


display_target_distribution(
    development_training_data,
    "Development training dataset"
)

display_target_distribution(
    development_validation_data,
    "Development validation dataset"
)

display_target_distribution(
    development_testing_data,
    "Development test dataset"
)

# Create feature sets

In [ ]:
# FINAL FINANCIAL AND HYBRID FEATURE SETS


MAXIMUM_MISSING_RATE = 0.50

def create_feature_sets(data):
    macro_features = [
        column for column in data.columns
        if column.startswith("macro_")
        and column not in EXCLUDED_MACRO_COLUMNS
    ]

    high_missing_columns = [
        column for column in data.columns
        if data[column].isna().mean() > MAXIMUM_MISSING_RATE
    ]
    constant_or_empty_columns = [
        column for column in data.columns
        if data[column].nunique(dropna=True) <= 1
    ]

    columns_to_exclude = set(existing_columns(
        data,
        IDENTIFIER_COLUMNS
        + POTENTIAL_LEAKAGE_COLUMNS
        + EMBEDDED_MACRO_COLUMNS
        + DATE_COLUMNS
        + high_missing_columns
        + constant_or_empty_columns
        + [TARGET_COLUMN]
    ))

    financial_features = [
        column for column in data.columns
        if column not in columns_to_exclude
        and not column.startswith("macro_")
    ]
    hybrid_features = financial_features + [
        column for column in macro_features
        if column not in high_missing_columns
    ]

    financial_features = list(dict.fromkeys(financial_features))
    hybrid_features = list(dict.fromkeys(hybrid_features))
    return financial_features, hybrid_features


# feature validation 

In [ ]:
# Validate the corrected feature sets using the already prepared H3 data
financial_features, hybrid_features = create_feature_sets(h3_prepared)

approved_macro_features = [
    column for column in hybrid_features
    if column.startswith("macro_")
]

forbidden_financial = [
    column for column in (
        POTENTIAL_LEAKAGE_COLUMNS
        + EMBEDDED_MACRO_COLUMNS
        + EXCLUDED_MACRO_COLUMNS
    )
    if column in financial_features
]

forbidden_hybrid = [
    column for column in (
        POTENTIAL_LEAKAGE_COLUMNS
        + EXCLUDED_MACRO_COLUMNS
    )
    if column in hybrid_features
]

print("Financial features:", len(financial_features))
print("Hybrid features:", len(hybrid_features))
print("Approved macroeconomic features:", approved_macro_features)
print("Forbidden financial features:", forbidden_financial)
print("Forbidden hybrid features:", forbidden_hybrid)

assert not forbidden_financial
assert not forbidden_hybrid
assert TARGET_COLUMN not in financial_features
assert COMPANY_ID not in financial_features

print("Feature leakage validation: PASSED")


# Audit operational status

In [ ]:
# OPERATIONAL-STATUS LEAKAGE AUDIT


if "operational_status" in h3_data.columns:

    status_audit = (
        h3_data
        .groupby("operational_status", dropna=False)
        [TARGET_COLUMN]
        .agg(
            observations="size",
            distressed_cases="sum",
            distress_rate="mean"
        )
        .sort_values(
            "distress_rate",
            ascending=False
        )
    )

    display(status_audit)

else:
    print("operational_status was not found.")

# Check whether macroeconomic values are genuinely lagged

# Define faster models

In [ ]:
# Build streamlined development models.
def create_fast_development_models(
    X_train
):

    logistic_regression = Pipeline(
        steps=[
            (
                "preprocessor",
                create_preprocessor(
                    X_train,
                    scale_numeric=True
                )
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=300,
                    solver="liblinear",
                    class_weight="balanced",
                    random_state=
                        RANDOM_STATE
                )
            )
        ]
    )

    xgboost_model = Pipeline(
        steps=[
            (
                "preprocessor",
                create_preprocessor(
                    X_train,
                    scale_numeric=False
                )
            ),
            (
                "model",
                XGBClassifier(
                    n_estimators=
                        XGBOOST_ESTIMATORS,
                    learning_rate=0.08,
                    max_depth=5,
                    min_child_weight=5,
                    subsample=0.80,
                    colsample_bytree=0.80,
                    objective=
                        "binary:logistic",
                    eval_metric="logloss",
                    tree_method="hist",
                    n_jobs=-1,
                    random_state=
                        RANDOM_STATE
                )
            )
        ]
    )

    return {
        "Logistic Regression":
            logistic_regression,

        "XGBoost":
            xgboost_model
    }

# Run the financial-only models

In [ ]:
# Evaluate the financial baseline models.
financial_development_results = []

selected_features = (
    financial_features
)

X_train_financial = (
    development_training_data[
        selected_features
    ]
)

y_train_financial = (
    development_training_data[
        TARGET_COLUMN
    ]
    .astype(int)
)

X_validation_financial = (
    development_validation_data[
        selected_features
    ]
)

y_validation_financial = (
    development_validation_data[
        TARGET_COLUMN
    ]
    .astype(int)
)

X_test_financial = (
    development_testing_data[
        selected_features
    ]
)

y_test_financial = (
    development_testing_data[
        TARGET_COLUMN
    ]
    .astype(int)
)

financial_models = (
    create_fast_development_models(
        X_train_financial
    )
)

for model_name, model in (
    financial_models.items()
):

    print(
        "\nFinancial-only model:",
        model_name
    )

    start_time = time.time()

    model.fit(
        X_train_financial,
        y_train_financial
    )

    validation_probability = (
        model.predict_proba(
            X_validation_financial
        )[:, 1]
    )

    best_threshold, validation_f1 = (
        select_threshold(
            y_validation_financial,
            validation_probability
        )
    )

    test_probability = (
        model.predict_proba(
            X_test_financial
        )[:, 1]
    )

    result = calculate_metrics(
        y_test_financial,
        test_probability,
        best_threshold
    )

    result.update(
        {
            "horizon": "H3",
            "years_ahead": 2,
            "feature_set": "financial",
            "model": model_name,
            "validation_f1":
                validation_f1,
            "fit_seconds":
                time.time() -
                start_time
        }
    )

    financial_development_results.append(
        result
    )

    print(
        "Completed in:",
        f"{result['fit_seconds']:.1f}",
        "seconds"
    )

    print(
        "Threshold:",
        round(
            result["threshold"],
            4
        )
    )

    print(
        "Test F1:",
        round(
            result["f1"],
            4
        )
    )

    print(
        "Test ROC-AUC:",
        round(
            result["roc_auc"],
            4
        )
    )

    print(
        "Test PR-AUC:",
        round(
            result["pr_auc"],
            4
        )
    )

    del model
    gc.collect()

# Run the hybrid models

In [ ]:
# Evaluate the hybrid feature models.
hybrid_development_results = []

selected_features = (
    hybrid_features
)

X_train_hybrid = (
    development_training_data[
        selected_features
    ]
)

y_train_hybrid = (
    development_training_data[
        TARGET_COLUMN
    ]
    .astype(int)
)

X_validation_hybrid = (
    development_validation_data[
        selected_features
    ]
)

y_validation_hybrid = (
    development_validation_data[
        TARGET_COLUMN
    ]
    .astype(int)
)

X_test_hybrid = (
    development_testing_data[
        selected_features
    ]
)

y_test_hybrid = (
    development_testing_data[
        TARGET_COLUMN
    ]
    .astype(int)
)

hybrid_models = (
    create_fast_development_models(
        X_train_hybrid
    )
)

for model_name, model in (
    hybrid_models.items()
):

    print(
        "\nHybrid model:",
        model_name
    )

    start_time = time.time()

    model.fit(
        X_train_hybrid,
        y_train_hybrid
    )

    validation_probability = (
        model.predict_proba(
            X_validation_hybrid
        )[:, 1]
    )

    best_threshold, validation_f1 = (
        select_threshold(
            y_validation_hybrid,
            validation_probability
        )
    )

    test_probability = (
        model.predict_proba(
            X_test_hybrid
        )[:, 1]
    )

    result = calculate_metrics(
        y_test_hybrid,
        test_probability,
        best_threshold
    )

    result.update(
        {
            "horizon": "H3",
            "years_ahead": 2,
            "feature_set": "hybrid",
            "model": model_name,
            "validation_f1":
                validation_f1,
            "fit_seconds":
                time.time() -
                start_time
        }
    )

    hybrid_development_results.append(
        result
    )

    print(
        "Completed in:",
        f"{result['fit_seconds']:.1f}",
        "seconds"
    )

    print(
        "Threshold:",
        round(
            result["threshold"],
            4
        )
    )

    print(
        "Test F1:",
        round(
            result["f1"],
            4
        )
    )

    print(
        "Test ROC-AUC:",
        round(
            result["roc_auc"],
            4
        )
    )

    print(
        "Test PR-AUC:",
        round(
            result["pr_auc"],
            4
        )
    )

    del model
    gc.collect()

# Combine and save H3 results

In [ ]:
# Combine development results for comparison.
development_results = pd.DataFrame(
    financial_development_results +
    hybrid_development_results
)

result_columns = [
    "horizon",
    "years_ahead",
    "feature_set",
    "model",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "brier_score",
    "validation_f1",
    "fit_seconds"
]

development_results = (
    development_results[
        result_columns
    ]
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    development_results
)

development_results.to_csv(
    results_folder /
    "h3_fast_development_results.csv",
    index=False
)

print(
    "H3 development results saved in:",
    results_folder /
    "h3_fast_development_results.csv"
)

# Optimised H1–H6 analysis

In [ ]:
# FINAL PREDICTIVE HORIZONS

# Main early-warning analysis
FINAL_HORIZONS = [2, 3, 4, 5, 6]

# H1 should be analysed and reported separately
CONTEMPORANEOUS_HORIZON = [1]

FINAL_FOLDS = [0, 1, 2, 3, 4]

print(
    "Primary forecasting horizons:",
    FINAL_HORIZONS
)

print(
    "Separate contemporaneous benchmark:",
    CONTEMPORANEOUS_HORIZON
)

In [ ]:

# MACRO AVAILABILITY AND TYPE VALIDATION

def validate_and_filter_horizon(data, horizon):
    """Prepare model-safe types and retain rows with lagged macro data."""

    boolean_columns = data.select_dtypes(
        include=["bool", "boolean"]
    ).columns.tolist()

    for column in boolean_columns:
        data[column] = data[column].astype("float32")

    approved_macro = [
        column for column in data.columns
        if column.startswith("macro_")
        and column not in EXCLUDED_MACRO_COLUMNS
    ]

    if not approved_macro:
        raise ValueError(f"H{horizon}: no approved macro variables were found.")

    missing_macro_mask = data[approved_macro].isna().all(axis=1)
    excluded_rows = int(missing_macro_mask.sum())
    availability = 100 * (1 - excluded_rows / len(data))

    print(f"H{horizon} Boolean columns converted: {boolean_columns}")
    print(f"H{horizon} approved macro features: {len(approved_macro)}")
    print(f"H{horizon} lagged macro availability: {availability:.2f}%")
    print(f"H{horizon} rows excluded without lagged macro data: {excluded_rows:,}")

    # Use the same complete sample for financial and hybrid models.
    data = data.loc[~missing_macro_mask].copy()

    return data, approved_macro


In [ ]:
# lARGE H3 dEVELOPMENT OBJECTS BEFORE H2-H6 ESTIMATION
for object_name in [
    "h3_data", "h3_prepared", "h3_model_data",
    "complete_training_data", "complete_validation_data",
    "complete_testing_data", "development_training_data",
    "development_validation_data", "development_testing_data",
    "X_train_financial", "X_validation_financial", "X_test_financial",
    "X_train_hybrid", "X_validation_hybrid", "X_test_hybrid"
]:
    if object_name in globals():
        del globals()[object_name]

gc.collect()
print("H3 development objects released from memory.")


In [ ]:
# CONTROLLED SAMPLING FOR FASTER MODEL ESTIMATION

TRAIN_NEGATIVE_RATIO = 5
MAX_VALIDATION_ROWS = 50_000
MAX_TEST_ROWS = 50_000
RANDOM_STATE = 42


def sample_training_data(data, target_column, negative_ratio=5):
    """
    Retain every distressed observation and sample a controlled
    number of non-distressed observations from the training fold.
    """

    positive_data = data.loc[
        data[target_column] == 1
    ]

    negative_data = data.loc[
        data[target_column] == 0
    ]

    required_negatives = min(
        len(negative_data),
        len(positive_data) * negative_ratio
    )

    sampled_negatives = negative_data.sample(
        n=required_negatives,
        random_state=RANDOM_STATE
    )

    sampled_data = pd.concat(
        [positive_data, sampled_negatives],
        ignore_index=False
    )

    return sampled_data.sample(
        frac=1,
        random_state=RANDOM_STATE
    )


def sample_evaluation_data(data, maximum_rows, seed):
    """
    Draw a reproducible evaluation sample without changing
    company-fold membership.
    """

    if len(data) <= maximum_rows:
        return data

    return data.sample(
        n=maximum_rows,
        random_state=seed
    )

In [ ]:

# FINAL H2-H6 MODEL TRAINING — OPTIMISED 

FINAL_HORIZONS = [2, 3, 4, 5, 6]
FINAL_FOLDS = [0, 1, 2, 3, 4]

predictions_folder = results_folder / "prediction_parts_corrected"
predictions_folder.mkdir(parents=True, exist_ok=True)

print("Horizons:", FINAL_HORIZONS)
print("Folds:", FINAL_FOLDS)

final_results = []

for horizon in FINAL_HORIZONS:
    print(f"\n========== H{horizon} ==========")

    data = prepare_horizon_data(horizon)
    data, current_macro_features = validate_and_filter_horizon(data, horizon)
    data["_fold"] = create_grouped_folds(data)

    financial_features, hybrid_features = create_feature_sets(data)
    feature_dictionary = {
        "financial": financial_features,
        "hybrid": hybrid_features
    }

    for validation_fold in FINAL_FOLDS:
        test_fold = (validation_fold + 1) % 5

        complete_training = data.loc[
            ~data["_fold"].isin([validation_fold, test_fold])
        ].copy()
        complete_validation = data.loc[
            data["_fold"] == validation_fold
        ].copy()
        complete_testing = data.loc[
            data["_fold"] == test_fold
        ].copy()

        training_data = sample_training_data(
            complete_training,
            TARGET_COLUMN,
            negative_ratio=TRAIN_NEGATIVE_RATIO
        )
        validation_data = sample_evaluation_data(
            complete_validation,
            MAX_VALIDATION_ROWS,
            seed=RANDOM_STATE + validation_fold
        )
        testing_data = sample_evaluation_data(
            complete_testing,
            MAX_TEST_ROWS,
            seed=RANDOM_STATE + 100 + validation_fold
        )

        print(
            f"Fold {validation_fold}: train={len(training_data):,}, "
            f"validation={len(validation_data):,}, test={len(testing_data):,}"
        )

        del complete_training, complete_validation, complete_testing
        gc.collect()

        for feature_set, features in feature_dictionary.items():
            X_train = training_data[features]
            y_train = training_data[TARGET_COLUMN].astype(int)
            X_validation = validation_data[features]
            y_validation = validation_data[TARGET_COLUMN].astype(int)
            X_test = testing_data[features]
            y_test = testing_data[TARGET_COLUMN].astype(int)

            models = create_fast_development_models(X_train)

            for model_name, model in models.items():
                print(
                    f"H{horizon} | fold {validation_fold} | "
                    f"{feature_set} | {model_name}"
                )
                start_time = time.time()
                model.fit(X_train, y_train)

                validation_probability = model.predict_proba(X_validation)[:, 1]
                threshold, validation_f1 = select_threshold(
                    y_validation,
                    validation_probability
                )
                test_probability = model.predict_proba(X_test)[:, 1]
                result = calculate_metrics(y_test, test_probability, threshold)
                result.update({
                    "horizon": f"H{horizon}",
                    "years_ahead": horizon - 1,
                    "validation_fold": validation_fold,
                    "test_fold": test_fold,
                    "feature_set": feature_set,
                    "model": model_name,
                    "validation_f1": validation_f1,
                    "training_rows": len(training_data),
                    "validation_rows": len(validation_data),
                    "testing_rows": len(testing_data),
                    "fit_seconds": time.time() - start_time
                })
                final_results.append(result)

                prediction_output = pd.DataFrame({
                    "horizon": f"H{horizon}",
                    "years_ahead": horizon - 1,
                    "validation_fold": validation_fold,
                    "test_fold": test_fold,
                    "feature_set": feature_set,
                    "model": model_name,
                    "emis_id": testing_data[COMPANY_ID].to_numpy(),
                    "country": testing_data["analysis_country_code"].to_numpy(),
                    "year": testing_data[YEAR_COLUMN].to_numpy(),
                    "actual": y_test.to_numpy(),
                    "probability": test_probability,
                    "predicted": (test_probability >= threshold).astype("int8"),
                    "threshold": threshold
                })

                safe_model_name = model_name.lower().replace(" ", "_")
                prediction_file = predictions_folder / (
                    f"h{horizon}_fold{validation_fold}_"
                    f"{feature_set}_{safe_model_name}.parquet"
                )
                prediction_output.to_parquet(prediction_file, index=False)

                del model, prediction_output, validation_probability, test_probability
                gc.collect()

            del models, X_train, X_validation, X_test
            gc.collect()

        del training_data, validation_data, testing_data
        gc.collect()

    del data
    gc.collect()

final_results = pd.DataFrame(final_results)

expected_rows = (
    len(FINAL_HORIZONS) * len(FINAL_FOLDS) * 2 * 2
)
assert len(final_results) == expected_rows

final_results.to_csv(
    results_folder / "corrected_forward_model_results.csv",
    index=False
)

print("\nFinal training completed.")
print("Final results dimensions:", final_results.shape)
print("Prediction files saved in:", predictions_folder)
display(final_results.head())


# Summarise performance and hybrid improvement

In [ ]:

# SUMMARISE CROSS-FOLD PERFORMANCE

if "final_results" not in globals() or final_results.empty:
    corrected_results_file = results_folder / "corrected_forward_model_results.csv"
    if corrected_results_file.exists():
        final_results = pd.read_csv(corrected_results_file)
    else:
        raise RuntimeError("Run the corrected H2-H6 training cell first.")

performance_metrics = [
    "accuracy", "balanced_accuracy", "precision", "recall",
    "f1", "roc_auc", "pr_auc", "brier_score"
]

mean_performance = (
    final_results
    .groupby(["horizon", "years_ahead", "feature_set", "model"], as_index=False)
    [performance_metrics]
    .mean()
)

display(mean_performance)
mean_performance.to_csv(
    results_folder / "mean_crossfold_performance.csv",
    index=False
)


# Financial versus hybrid comparison


In [ ]:
# FINANCIAL VERSUS HYBRID COMPARISON

comparison = (
    final_results
    .groupby(["horizon", "years_ahead", "feature_set", "model"], as_index=False)
    .agg(
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        mean_roc_auc=("roc_auc", "mean"),
        mean_pr_auc=("pr_auc", "mean"),
        sd_pr_auc=("pr_auc", "std")
    )
    .sort_values(["horizon", "model", "feature_set"])
)

display(comparison)
comparison.to_csv(
    results_folder / "corrected_financial_hybrid_comparison.csv",
    index=False
)


# Macroeconomic Improvement

In [ ]:
# INCREMENTAL VALUE OF MACROECONOMIC VARIABLES

metric_comparison = comparison.pivot_table(
    index=["horizon", "years_ahead", "model"],
    columns="feature_set",
    values=["mean_f1", "mean_recall", "mean_roc_auc", "mean_pr_auc"]
)

metric_comparison.columns = [
    f"{metric}_{feature_set}"
    for metric, feature_set in metric_comparison.columns
]
metric_comparison = metric_comparison.reset_index()

for metric in ["mean_f1", "mean_recall", "mean_roc_auc", "mean_pr_auc"]:
    financial_column = f"{metric}_financial"
    hybrid_column = f"{metric}_hybrid"
    metric_comparison[f"{metric}_hybrid_improvement"] = (
        metric_comparison[hybrid_column]
        - metric_comparison[financial_column]
    )

display(metric_comparison)
metric_comparison.to_csv(
    results_folder / "corrected_macroeconomic_improvement.csv",
    index=False
)


# Best forward model

In [ ]:
# SELECT BEST FORWARD-LOOKING MODEL USING MEAN CROSS-FOLD PR-AUC

best_forward_model = (
    comparison
    .sort_values(["mean_pr_auc", "mean_f1", "mean_recall"], ascending=False)
    .iloc[0]
)

display(best_forward_model.to_frame("value"))
print(
    "Selected best forward-looking model:",
    best_forward_model["horizon"],
    best_forward_model["feature_set"],
    best_forward_model["model"]
)


# Plot performance across horizons

In [ ]:
# Plot performance across forecast horizons.
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

for axis, metric in zip(
    axes,
    [
        "pr_auc",
        "f1",
        "roc_auc"
    ]
):

    sns.lineplot(
        data=mean_performance,
        x="years_ahead",
        y=metric,
        hue="model",
        style="feature_set",
        markers=True,
        dashes=True,
        ax=axis
    )

    axis.set_title(
        f"{metric.upper()} Across Horizons"
    )

    axis.set_xlabel("Years ahead")
    axis.set_ylabel(metric.upper())

plt.tight_layout()

plt.savefig(
    results_folder / "performance_across_horizons.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Interpretation and robustness analysis

In [ ]:
# LOAD SELECTED-MODEL OUT-OF-FOLD PREDICTIONS

predictions_folder = results_folder / "prediction_parts_corrected"
selected_horizon_number = int(str(best_forward_model["horizon"]).replace("H", ""))
selected_feature_set = str(best_forward_model["feature_set"])
selected_model_name = str(best_forward_model["model"])
safe_model_name = selected_model_name.lower().replace(" ", "_")

prediction_files = sorted(predictions_folder.glob(
    f"h{selected_horizon_number}_fold*_{selected_feature_set}_{safe_model_name}.parquet"
))

print("Selected prediction files found:", len(prediction_files))
if len(prediction_files) != 5:
    raise FileNotFoundError(
        "Expected five selected-model prediction files. "
        "Run the corrected H2-H6 training cell first."
    )

all_predictions = pd.concat(
    [pd.read_parquet(file) for file in prediction_files],
    ignore_index=True
)

required_prediction_columns = [
    "horizon", "feature_set", "model", "country",
    "actual", "probability", "predicted"
]
missing_prediction_columns = [
    column for column in required_prediction_columns
    if column not in all_predictions.columns
]
if missing_prediction_columns:
    raise KeyError(f"Missing prediction columns: {missing_prediction_columns}")

print("Combined selected predictions:", all_predictions.shape)
display(all_predictions.head())


# Country Level Performance

In [ ]:
# Calculate performance for each country.
country_results = []

for keys, group in all_predictions.groupby(
    [
        "horizon",
        "feature_set",
        "model",
        "country"
    ]
):

    horizon = keys[0]
    feature_set = keys[1]
    model_name = keys[2]
    country = keys[3]

    # ROC-AUC and PR-AUC require both target classes
    if group["actual"].nunique() < 2:
        continue

    country_results.append(
        {
            "horizon":
                horizon,

            "feature_set":
                feature_set,

            "model":
                model_name,

            "country":
                country,

            "observations":
                len(group),

            "distressed_cases":
                int(
                    group["actual"].sum()
                ),

            "distress_rate":
                group["actual"].mean(),

            "precision":
                precision_score(
                    group["actual"],
                    group["predicted"],
                    zero_division=0
                ),

            "recall":
                recall_score(
                    group["actual"],
                    group["predicted"],
                    zero_division=0
                ),

            "f1":
                f1_score(
                    group["actual"],
                    group["predicted"],
                    zero_division=0
                ),

            "roc_auc":
                roc_auc_score(
                    group["actual"],
                    group["probability"]
                ),

            "pr_auc":
                average_precision_score(
                    group["actual"],
                    group["probability"]
                )
        }
    )

country_performance = pd.DataFrame(
    country_results
)

if country_performance.empty:

    raise ValueError(
        "No country groups contained both target classes."
    )

country_performance = (
    country_performance
    .sort_values(
        [
            "horizon",
            "feature_set",
            "model",
            "country"
        ]
    )
    .reset_index(drop=True)
)

display(
    country_performance.head(50)
)

country_performance.to_csv(
    results_folder /
    "performance_by_country.csv",
    index=False
)

print(
    "Country-level results saved successfully."
)

# Risk-decile analysis

In [ ]:
# USE THE BEST MODEL SELECTED FROM MEAN CROSS-FOLD PR-AUC
best_result = best_forward_model

print("Selected best forward-looking model:")
display(best_result.to_frame("value"))

best_predictions = all_predictions.loc[
    (all_predictions["horizon"].astype(str) == str(best_result["horizon"]))
    & (all_predictions["feature_set"].astype(str) == str(best_result["feature_set"]))
    & (all_predictions["model"].astype(str) == str(best_result["model"]))
].copy()

print("Selected pooled out-of-fold prediction rows:", f"{len(best_predictions):,}")
if best_predictions.empty:
    raise ValueError("No predictions matched the selected best model.")

display(
    best_predictions["actual"]
    .value_counts()
    .sort_index()
    .rename_axis("actual")
    .reset_index(name="observations")
)

best_predictions["risk_decile"] = pd.qcut(
    best_predictions["probability"].rank(method="first"),
    q=10,
    labels=range(1, 11)
)

risk_deciles = (
    best_predictions
    .groupby("risk_decile", observed=True)
    .agg(
        observations=("actual", "size"),
        distressed_cases=("actual", "sum"),
        observed_distress_rate=("actual", "mean"),
        minimum_predicted_risk=("probability", "min"),
        mean_predicted_risk=("probability", "mean"),
        maximum_predicted_risk=("probability", "max")
    )
    .reset_index()
)

display(risk_deciles)
risk_deciles.to_csv(
    results_folder / "risk_decile_analysis.csv",
    index=False
)

plt.figure(figsize=(10, 6))
plt.plot(
    risk_deciles["risk_decile"],
    risk_deciles["observed_distress_rate"],
    marker="o",
    linewidth=2,
    label="Observed distress rate"
)
plt.plot(
    risk_deciles["risk_decile"],
    risk_deciles["mean_predicted_risk"],
    marker="s",
    linewidth=2,
    label="Mean predicted risk"
)
plt.title("Observed and Predicted Distress by Risk Decile")
plt.xlabel("Predicted-risk decile")
plt.ylabel("Distress rate")
plt.xticks(range(1, 11))
plt.legend()
plt.tight_layout()
plt.savefig(
    results_folder / "risk_decile_analysis.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print("Risk-decile analysis saved successfully.")


# Retrain H2 hybrid XGBoost model for SHAP




In [ ]:
# Configure the SHAP analysis sample.
SHAP_HORIZON = 2
SHAP_VALIDATION_FOLD = 0

shap_data = prepare_horizon_data(SHAP_HORIZON)
shap_data, shap_macro_features = validate_and_filter_horizon(
    shap_data,
    SHAP_HORIZON
)
shap_data["_fold"] = create_grouped_folds(shap_data)

shap_test_fold = (SHAP_VALIDATION_FOLD + 1) % 5
shap_training_data = shap_data.loc[
    ~shap_data["_fold"].isin([SHAP_VALIDATION_FOLD, shap_test_fold])
].copy()
shap_test_data = shap_data.loc[
    shap_data["_fold"] == shap_test_fold
].copy()

shap_training_data = sample_training_data(
    shap_training_data,
    TARGET_COLUMN,
    negative_ratio=TRAIN_NEGATIVE_RATIO
)

financial_features, hybrid_features = create_feature_sets(shap_data)
X_shap_train = shap_training_data[hybrid_features]
y_shap_train = shap_training_data[TARGET_COLUMN].astype(int)

shap_model = create_fast_development_models(X_shap_train)["XGBoost"]
shap_model.fit(X_shap_train, y_shap_train)

X_shap_test = shap_test_data[hybrid_features]
explanation_rows = min(2_000, len(X_shap_test))
X_explain = X_shap_test.sample(
    n=explanation_rows,
    random_state=RANDOM_STATE
)

preprocessor = shap_model.named_steps["preprocessor"]
xgb_model = shap_model.named_steps["model"]

print("SHAP model trained successfully.")
print("SHAP explanation rows:", explanation_rows)


#  Global and local SHAP explanations

In [ ]:
# TRANSFORM DATA AND CALCULATE SHAP VALUES

X_transformed = preprocessor.transform(X_explain)

numeric_input_features = list(preprocessor.transformers_[0][2])
numeric_imputer = (
    preprocessor.named_transformers_["numeric"].named_steps["imputer"]
)
numeric_feature_names = [
    feature for feature, statistic
    in zip(numeric_input_features, numeric_imputer.statistics_)
    if not pd.isna(statistic)
]

categorical_input_features = list(preprocessor.transformers_[1][2])
categorical_feature_names = []
if categorical_input_features:
    fitted_categorical_pipeline = preprocessor.named_transformers_["categorical"]
    fitted_encoder = fitted_categorical_pipeline.named_steps["onehot"]
    if hasattr(fitted_encoder, "categories_"):
        categorical_feature_names = list(
            fitted_encoder.get_feature_names_out(categorical_input_features)
        )

feature_names = np.asarray(
    numeric_feature_names + categorical_feature_names,
    dtype=object
)

if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()
X_transformed = np.asarray(X_transformed, dtype="float32")

if len(feature_names) != X_transformed.shape[1]:
    raise ValueError(
        f"Feature-name mismatch: {len(feature_names)} names for "
        f"{X_transformed.shape[1]} transformed columns."
    )

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_transformed)
shap_values.feature_names = list(feature_names)

print("SHAP feature-name validation: PASSED")
print("SHAP dimensions:", shap_values.values.shape)


# Global SHAP plot

In [ ]:
# Display global SHAP effect patterns.
shap.plots.beeswarm(
    shap_values,
    max_display=20,
    show=False
)

plt.title(
    f"H{SHAP_HORIZON} Hybrid XGBoost: Global SHAP Importance"
)

plt.tight_layout()

plt.savefig(
    results_folder /
    f"h{SHAP_HORIZON}_global_shap_beeswarm.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# SHAP IMPORTANCE TABLE 

In [ ]:
# Rank features by SHAP importance.
shap_importance = pd.DataFrame(
    {
        "feature":
            feature_names,

        "mean_absolute_shap":
            np.abs(
                shap_values.values
            ).mean(axis=0)
    }
).sort_values(
    "mean_absolute_shap",
    ascending=False
)

shap_importance[
    "feature_group"
] = np.where(
    shap_importance[
        "feature"
    ].astype(str).str.startswith(
        "macro_"
    ),
    "Macroeconomic",
    "Financial or firm-level"
)

display(
    shap_importance.head(30)
)

shap_importance.to_csv(
    results_folder /
    f"h{SHAP_HORIZON}_shap_feature_importance.csv",
    index=False
)

# Save final model and configuration

In [ ]:
# Save the fitted model pipeline.
joblib.dump(
    {
        "model": shap_model,
        "features": hybrid_features,
        "horizon": SHAP_HORIZON,
        "company_identifier": COMPANY_ID,
        "target": TARGET_COLUMN,
        "macroeconomic_variables": shap_macro_features
    },
    results_folder / f"selected_h{SHAP_HORIZON}_hybrid_xgboost_model.joblib"
)

run_configuration = {
    "random_state": RANDOM_STATE,
    "horizons": FINAL_HORIZONS,
    "company_identifier": COMPANY_ID,
    "country_identifier": "analysis_country_code",
    "target": TARGET_COLUMN,
    "models": ["Logistic Regression", "XGBoost"],
    "feature_sets": ["financial", "hybrid"],
    "macroeconomic_variables": shap_macro_features,
    "excluded_leakage_variables": POTENTIAL_LEAKAGE_COLUMNS,
    "outlier_treatment": "Training-only 1st and 99th percentile winsorisation",
    "missing_value_treatment": "Training-set median and most-frequent imputation",
    "validation": "Five company-grouped and country-preserving folds",
    "training_sampling": "All positive observations and five negatives per positive",
    "evaluation_sampling": "Reproducible uniform sample of at most 50,000 rows per fold"
}

with open(
    results_folder / "run_configuration.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(run_configuration, file, indent=2)

print("Analysis completed.")
print("Results saved in:", results_folder)


In [ ]:
# LOCAL SHAP EXPLANATION AND CONTRIBUTION TABLE

explanation_probabilities = shap_model.predict_proba(X_explain)[:, 1]
local_position = int(np.argmax(explanation_probabilities))
local_probability = float(explanation_probabilities[local_position])

shap.plots.waterfall(
    shap_values[local_position],
    max_display=20,
    show=False
)
plt.title(
    f"H{SHAP_HORIZON} Local SHAP Explanation: "
    f"Predicted Risk {local_probability:.2%}"
)
plt.tight_layout()
plt.savefig(
    results_folder / f"h{SHAP_HORIZON}_local_shap_waterfall_highest_risk.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

local_shap_table = pd.DataFrame({
    "feature": feature_names,
    "transformed_value": X_transformed[local_position],
    "shap_contribution": shap_values.values[local_position]
})
local_shap_table["absolute_contribution"] = (
    local_shap_table["shap_contribution"].abs()
)
local_shap_table["effect_on_distress_prediction"] = np.where(
    local_shap_table["shap_contribution"] > 0,
    "Increases predicted distress",
    "Decreases predicted distress"
)
local_shap_table = local_shap_table.sort_values(
    "absolute_contribution",
    ascending=False
).reset_index(drop=True)

display(local_shap_table.head(20))
local_shap_table.to_csv(
    results_folder / f"h{SHAP_HORIZON}_local_shap_highest_risk_table.csv",
    index=False
)
print("Local SHAP explanation saved successfully.")


# Additional robustness and validation analyses



## 95% confidence intervals across grouped folds

.

In [ ]:
# CROSS-FOLD MEANS, STANDARD DEVIATIONS AND 95% INTERVALS

from scipy.stats import t as student_t

if "final_results" not in globals() or final_results.empty:
    results_file = results_folder / "corrected_forward_model_results.csv"
    final_results = pd.read_csv(results_file)

uncertainty_metrics = [
    "precision", "recall", "f1", "roc_auc", "pr_auc", "brier_score"
]
group_columns = ["horizon", "years_ahead", "feature_set", "model"]
uncertainty_rows = []

for keys, group in final_results.groupby(group_columns):
    for metric in uncertainty_metrics:
        values = group[metric].dropna().astype(float)
        number_of_folds = len(values)
        mean_value = values.mean()
        standard_deviation = values.std(ddof=1)
        standard_error = standard_deviation / np.sqrt(number_of_folds)
        critical_value = student_t.ppf(0.975, df=number_of_folds - 1)
        margin = critical_value * standard_error

        uncertainty_rows.append({
            **dict(zip(group_columns, keys)),
            "metric": metric,
            "folds": number_of_folds,
            "mean": mean_value,
            "standard_deviation": standard_deviation,
            "ci_95_lower": mean_value - margin,
            "ci_95_upper": mean_value + margin
        })

crossfold_uncertainty = pd.DataFrame(uncertainty_rows)
crossfold_uncertainty.to_csv(
    results_folder / "crossfold_95_percent_intervals.csv",
    index=False
)

selected_uncertainty = crossfold_uncertainty.loc[
    (crossfold_uncertainty["horizon"] == "H2")
    & (crossfold_uncertainty["feature_set"] == "hybrid")
    & (crossfold_uncertainty["model"] == "XGBoost")
].reset_index(drop=True)

display(selected_uncertainty)
print("Cross-fold uncertainty table saved successfully.")

## H2 probability-calibration assessment

The reliability plot compares predicted probabilities with observed distress rates for the pooled out-of-fold H2 predictions.

In [ ]:

# H2 RELIABILITY TABLE AND CALIBRATION PLOT

predictions_folder = results_folder / "prediction_parts_corrected"

if "all_predictions" not in globals() or all_predictions.empty:
    selected_files = sorted(
        predictions_folder.glob("h2_fold*_hybrid_xgboost.parquet")
    )
    if not selected_files:
        raise FileNotFoundError(
            "Run the final H2-H6 training cell before calibration analysis."
        )
    all_predictions = pd.concat(
        [pd.read_parquet(file) for file in selected_files],
        ignore_index=True
    )

h2_calibration_data = all_predictions.loc[
    (all_predictions["horizon"] == "H2")
    & (all_predictions["feature_set"] == "hybrid")
    & (all_predictions["model"] == "XGBoost"),
    ["actual", "probability"]
].copy()

h2_calibration_data["calibration_bin"] = pd.qcut(
    h2_calibration_data["probability"].rank(method="first"),
    q=10,
    labels=range(1, 11)
)

calibration_table = (
    h2_calibration_data
    .groupby("calibration_bin", observed=True)
    .agg(
        observations=("actual", "size"),
        mean_predicted_probability=("probability", "mean"),
        observed_distress_rate=("actual", "mean")
    )
    .reset_index()
)

calibration_table["absolute_calibration_gap"] = (
    calibration_table["mean_predicted_probability"]
    - calibration_table["observed_distress_rate"]
).abs()

expected_calibration_error = np.average(
    calibration_table["absolute_calibration_gap"],
    weights=calibration_table["observations"]
)

h2_brier_score = brier_score_loss(
    h2_calibration_data["actual"],
    h2_calibration_data["probability"]
)

calibration_table.to_csv(
    results_folder / "h2_calibration_table.csv",
    index=False
)

display(calibration_table)
print(f"H2 pooled Brier score: {h2_brier_score:.6f}")
print(f"H2 expected calibration error: {expected_calibration_error:.6f}")

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], "--", color="grey", label="Perfect calibration")
plt.plot(
    calibration_table["mean_predicted_probability"],
    calibration_table["observed_distress_rate"],
    marker="o",
    color="#3569A8",
    label="H2 hybrid XGBoost"
)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed distress rate")
plt.title("H2 Hybrid XGBoost: Probability Calibration")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(
    results_folder / "h2_probability_calibration.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## H2 classification-threshold sensitivity analysis


In [ ]:
# H2 CLASSIFICATION-THRESHOLD SENSITIVITY

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix
)

predictions_folder = results_folder / "prediction_parts_corrected"

if "all_predictions" not in globals() or all_predictions.empty:
    selected_files = sorted(
        predictions_folder.glob("h2_fold*_hybrid_xgboost.parquet")
    )
    if not selected_files:
        raise FileNotFoundError(
            "The saved H2 hybrid XGBoost prediction files were not found."
        )
    all_predictions = pd.concat(
        [pd.read_parquet(file) for file in selected_files],
        ignore_index=True
    )

threshold_data = all_predictions.loc[
    (all_predictions["horizon"] == "H2")
    & (all_predictions["feature_set"] == "hybrid")
    & (all_predictions["model"] == "XGBoost")
].copy()

threshold_grid = np.round(np.arange(0.05, 0.951, 0.05), 2)
threshold_rows = []
actual = threshold_data["actual"].astype(int).to_numpy()
probability = threshold_data["probability"].astype(float).to_numpy()

for threshold in threshold_grid:
    predicted = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        actual,
        predicted,
        labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            actual, predicted, zero_division=0
        ),
        "recall": recall_score(
            actual, predicted, zero_division=0
        ),
        "f1": f1_score(
            actual, predicted, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            actual, predicted
        ),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    })

threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_sensitivity.to_csv(
    results_folder / "h2_threshold_sensitivity.csv",
    index=False
)

display(threshold_sensitivity)

best_sensitivity_row = threshold_sensitivity.loc[
    threshold_sensitivity["f1"].idxmax()
]
print(
    "Highest pooled sensitivity-analysis F1 threshold:",
    f"{best_sensitivity_row['threshold']:.2f}"
)
print(
    "This descriptive threshold does not replace the "
    "validation-fold thresholds in the main results."
)

plt.figure(figsize=(8, 5))
plt.plot(
    threshold_sensitivity["threshold"],
    threshold_sensitivity["precision"],
    marker="o",
    label="Precision"
)
plt.plot(
    threshold_sensitivity["threshold"],
    threshold_sensitivity["recall"],
    marker="o",
    label="Recall"
)
plt.plot(
    threshold_sensitivity["threshold"],
    threshold_sensitivity["f1"],
    marker="o",
    label="F1"
)
plt.xlabel("Classification threshold")
plt.ylabel("Score")
plt.title("H2 Hybrid XGBoost: Threshold Sensitivity")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(
    results_folder / "h2_threshold_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
print("H2 threshold-sensitivity analysis saved successfully.")

## Additional visualisations



### Selected-feature correlation heatmap
.


In [ ]:
# SELECTED H2 FINANCIAL AND MACROECONOMIC CORRELATION HEATMAP

required_correlation_objects = [
    "shap_data", "shap_importance", "numeric_input_features",
    "shap_macro_features"
]
missing_correlation_objects = [
    name for name in required_correlation_objects if name not in globals()
]
if missing_correlation_objects:
    raise RuntimeError(
        "Run the H2 SHAP preparation and importance cells first. "
        f"Missing objects: {missing_correlation_objects}"
    )

ranked_numeric_features = [
    feature
    for feature in shap_importance["feature"].tolist()
    if feature in numeric_input_features and feature in shap_data.columns
]

selected_financial_features = [
    feature
    for feature in ranked_numeric_features
    if feature not in shap_macro_features
][:15]

selected_macro_features = [
    feature for feature in shap_macro_features
    if feature in shap_data.columns
]

correlation_features = list(dict.fromkeys(
    selected_financial_features + selected_macro_features
))

if len(correlation_features) < 2:
    raise ValueError("Fewer than two numeric features were available for correlation.")

correlation_sample = shap_data[correlation_features].sample(
    n=min(50_000, len(shap_data)),
    random_state=RANDOM_STATE
)
selected_feature_correlation = correlation_sample.corr(method="spearman")

plt.figure(figsize=(14, 11))
sns.heatmap(
    selected_feature_correlation,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 7},
    square=False,
    cbar_kws={"label": "Spearman correlation"}
)
plt.title("H2 Selected Financial and Macroeconomic Feature Correlations")
plt.xticks(rotation=55, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(
    results_folder / "h2_selected_feature_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

selected_feature_correlation.to_csv(
    results_folder / "h2_selected_feature_correlations.csv"
)
print("Features included:", len(correlation_features))
display(selected_feature_correlation.round(3))


### H2 ROC and precision–recall curves



In [ ]:
# POOLED OUT-OF-FOLD H2 ROC AND PRECISION-RECALL CURVES

from sklearn.metrics import roc_curve, precision_recall_curve, auc

predictions_folder = results_folder / "prediction_parts_corrected"
h2_prediction_files = sorted(predictions_folder.glob("h2_fold*_*.parquet"))

if not h2_prediction_files:
    raise FileNotFoundError(
        "No saved H2 prediction files were found. "
        "Run the final H2-H6 training cell first."
    )

h2_curve_predictions = pd.concat(
    [pd.read_parquet(file) for file in h2_prediction_files],
    ignore_index=True
)

h2_curve_predictions = h2_curve_predictions.loc[
    h2_curve_predictions["horizon"].astype(str).eq("H2")
].copy()

expected_configurations = {
    ("financial", "Logistic Regression"),
    ("hybrid", "Logistic Regression"),
    ("financial", "XGBoost"),
    ("hybrid", "XGBoost")
}
available_configurations = set(
    h2_curve_predictions[["feature_set", "model"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
missing_configurations = expected_configurations - available_configurations
if missing_configurations:
    raise ValueError(f"Missing H2 model configurations: {missing_configurations}")

curve_colours = {
    ("financial", "Logistic Regression"): "#7A7A7A",
    ("hybrid", "Logistic Regression"): "#4C78A8",
    ("financial", "XGBoost"): "#F58518",
    ("hybrid", "XGBoost"): "#2E8B57"
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
curve_summary_rows = []

for (feature_set, model_name), group in h2_curve_predictions.groupby(
    ["feature_set", "model"], sort=False
):
    actual = group["actual"].astype(int).to_numpy()
    score = group["probability"].astype(float).to_numpy()
    label = f"{model_name} — {feature_set.title()}"
    colour = curve_colours[(feature_set, model_name)]

    false_positive_rate, true_positive_rate, _ = roc_curve(actual, score)
    roc_value = roc_auc_score(actual, score)
    axes[0].plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        color=colour,
        label=f"{label} (AUC={roc_value:.3f})"
    )

    precision_values, recall_values, _ = precision_recall_curve(actual, score)
    pr_value = average_precision_score(actual, score)
    axes[1].plot(
        recall_values,
        precision_values,
        linewidth=2,
        color=colour,
        label=f"{label} (AP={pr_value:.3f})"
    )

    curve_summary_rows.append({
        "feature_set": feature_set,
        "model": model_name,
        "observations": len(group),
        "distress_rate": actual.mean(),
        "roc_auc": roc_value,
        "pr_auc": pr_value
    })

axes[0].plot([0, 1], [0, 1], "--", color="black", alpha=0.6, label="No skill")
axes[0].set_title("H2 ROC Curves")
axes[0].set_xlabel("False-positive rate")
axes[0].set_ylabel("True-positive rate")
axes[0].legend(fontsize=8)

no_skill_precision = h2_curve_predictions["actual"].mean()
axes[1].axhline(
    no_skill_precision,
    linestyle="--",
    color="black",
    alpha=0.6,
    label=f"No skill ({no_skill_precision:.4f})"
)
axes[1].set_title("H2 Precision–Recall Curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(fontsize=8)

for axis in axes:
    axis.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(
    results_folder / "h2_roc_and_precision_recall_curves.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

h2_curve_summary = pd.DataFrame(curve_summary_rows).sort_values(
    ["pr_auc", "roc_auc"], ascending=False
)
h2_curve_summary.to_csv(
    results_folder / "h2_curve_summary.csv",
    index=False
)
display(h2_curve_summary)


### H2 hybrid XGBoost confusion matrices



In [ ]:
# H2 HYBRID XGBOOST CONFUSION MATRICES

h2_hybrid_xgb = h2_curve_predictions.loc[
    (h2_curve_predictions["feature_set"] == "hybrid")
    & (h2_curve_predictions["model"] == "XGBoost")
].copy()

if h2_hybrid_xgb.empty:
    raise ValueError("H2 hybrid XGBoost predictions were not found.")

h2_confusion_counts = confusion_matrix(
    h2_hybrid_xgb["actual"].astype(int),
    h2_hybrid_xgb["predicted"].astype(int),
    labels=[0, 1]
)

h2_confusion_normalised = confusion_matrix(
    h2_hybrid_xgb["actual"].astype(int),
    h2_hybrid_xgb["predicted"].astype(int),
    labels=[0, 1],
    normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(
    h2_confusion_counts,
    annot=True,
    fmt=",d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Non-distress", "Distress"],
    yticklabels=["Non-distress", "Distress"],
    ax=axes[0]
)
axes[0].set_title("Confusion Matrix — Counts")
axes[0].set_xlabel("Predicted class")
axes[0].set_ylabel("Actual class")

sns.heatmap(
    h2_confusion_normalised,
    annot=True,
    fmt=".1%",
    cmap="Greens",
    vmin=0,
    vmax=1,
    cbar=False,
    xticklabels=["Non-distress", "Distress"],
    yticklabels=["Non-distress", "Distress"],
    ax=axes[1]
)
axes[1].set_title("Confusion Matrix — Within Actual Class")
axes[1].set_xlabel("Predicted class")
axes[1].set_ylabel("Actual class")

plt.suptitle("H2 Hybrid XGBoost at Validation-Selected Thresholds", y=1.03)
plt.tight_layout()
plt.savefig(
    results_folder / "h2_hybrid_xgboost_confusion_matrices.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

tn, fp, fn, tp = h2_confusion_counts.ravel()
h2_confusion_summary = pd.DataFrame({
    "measure": [
        "true_negatives", "false_positives",
        "false_negatives", "true_positives"
    ],
    "count": [tn, fp, fn, tp]
})
h2_confusion_summary.to_csv(
    results_folder / "h2_hybrid_xgboost_confusion_matrix.csv",
    index=False
)
display(h2_confusion_summary)
